# Text Summarization Experiment with PIXIU's FLARE Dataset


## Thử nghiệm một tác vụ quan trọng trong FLARE – đó là Text Summarization, bao gồm hai tập dữ liệu:

- ECTSUM – tóm tắt dạng trích xuất từ transcript họp báo cáo tài chính

- EDTSUM – tóm tắt dạng sinh (abstractive) từ các bài viết tin tức tài chính

Thông qua việc sử dụng các mô hình tóm tắt có sẵn như BART và sentence-BERT, em sẽ tiến hành thực nghiệm, phân tích kết quả và rút ra một số nhận xét về hiệu quả của mô hình trên các bài toán tóm tắt trong lĩnh vực tài chính.

In [45]:
  !pip install transformers rouge-score datasets --quiet


In [46]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
from rouge_score import rouge_scorer
from datasets import Dataset


In [47]:
from google.colab import files
import pandas as pd

# Hiển thị cửa sổ chọn file
uploaded = files.upload()

# Giả sử bạn vừa upload 1 file CSV duy nhất, lấy tên file:
filename = list(uploaded.keys())[0]

# Đọc file CSV thành DataFrame
df = pd.read_csv(filename)

print(f"Đã tải lên file: {filename}")
print("Kích thước DataFrame:", df.shape)
df.head()

Saving flare-edtsum.csv to flare-edtsum.csv
Đã tải lên file: flare-edtsum.csv
Kích thước DataFrame: (2000, 4)


,id,query,answer,text
0,edtsum0,You are given a text that consists of multiple...,All-season Tire Market in Europe to Reach USD ...,LONDON--(BUSINESS WIRE)--Technavio has been mo...
1,edtsum1,You are given a text that consists of multiple...,Loop Energy Applauds Skywell and Joint Venture...,Achievementis an important step in expanding L...
2,edtsum2,You are given a text that consists of multiple...,"Chocolate Market - Growth, Trends, and Forecas...",LONDON--(BUSINESS WIRE)--The chocolate market ...
3,edtsum3,You are given a text that consists of multiple...,ABC Technologies Holdings Inc. Files Final Pro...,TORONTO--(BUSINESS WIRE)--ABC Technologies Hol...
4,edtsum4,You are given a text that consists of multiple...,FDJ: 2021 Financial Communication Calendar,"BOULOGNE-BILLANCOURT, France--(BUSINESS WIRE)-..."


In [48]:
from google.colab import files
import pandas as pd

# Hiển thị cửa sổ chọn file
uploaded = files.upload()

# Giả sử bạn vừa upload 1 file CSV duy nhất, lấy tên file:
filename = list(uploaded.keys())[0]

# Đọc file CSV thành DataFrame
df = pd.read_csv(filename)

print(f"Đã tải lên file: {filename}")
print("Kích thước DataFrame:", df.shape)
df.head()

Saving flare-ectsum.csv to flare-ectsum.csv
Đã tải lên file: flare-ectsum.csv
Kích thước DataFrame: (495, 5)


,id,query,answer,label,text
0,ectsum0,"Given the following article, please produce a ...",0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n1...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",During hearings spring [Phonetic] resulted in ...
1,ectsum1,"Given the following article, please produce a ...",1\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",And I'm confident that our disciplined approac...
2,ectsum2,"Given the following article, please produce a ...",0\n0\n0\n1\n0\n0\n0\n0\n1\n0\n0\n0\n0\n0\n0\n1...,"[0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...",Our fourth quarter results are highlighted by ...
3,ectsum3,"Given the following article, please produce a ...",1\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","In the first quarter, Cullen/Frost earned $113..."
4,ectsum4,"Given the following article, please produce a ...",1\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","Net earnings reached a record $1.2 billion, or..."


In [49]:
# Load extractive and abstractive summarization datasets
ectsum_df = pd.read_csv("flare-ectsum.csv")
edtsum_df = pd.read_csv("flare-edtsum.csv")


# Chọn mô hình tóm tắt

In [50]:
# Using pretrained BART for summarization
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Device set to use cuda:0


# Tóm tắt với EDTSUM (Abstractive)

In [52]:
sample_texts = [text[:1000] for text in edtsum_df["text"].head(5) if isinstance(text, str) and text.strip()]
reference_summaries = edtsum_df["answer"].head(len(sample_texts)).tolist()

generated_summaries = []
for text in sample_texts:
    try:
        result = summarizer(text, max_length=60, min_length=10, do_sample=False)
        generated_summaries.append(result[0]['summary_text'])
    except Exception as e:
        generated_summaries.append(f"Lỗi: {e}")


# Đánh giá bằng ROUGE

In [53]:
# Evaluate ROUGE scores
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
scores = [scorer.score(ref, pred) for ref, pred in zip(reference_summaries, generated_summaries)]

# Trung bình
avg_scores = {
    "rouge1": sum(s["rouge1"].fmeasure for s in scores) / len(scores),
    "rouge2": sum(s["rouge2"].fmeasure for s in scores) / len(scores),
    "rougeL": sum(s["rougeL"].fmeasure for s in scores) / len(scores),
}

print("ROUGE Scores (EDTSUM, abstractive):", avg_scores)


ROUGE Scores (EDTSUM, abstractive): {'rouge1': 0.03054720273230129, 'rouge2': 0.0, 'rougeL': 0.020791105171325687}


# Xử lý ECTSUM (Extractive)

In [55]:
!pip install sentence-transformers --quiet


In [58]:
from sentence_transformers import SentenceTransformer

model_sbert = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')  # ép chạy trên CPU

sample_row = ectsum_df.iloc[0]
sentences = sample_row["text"].split("\n")
reference_labels = list(map(int, sample_row["label"].strip("[]").split(",")))

# Sinh embedding
embeddings = model_sbert.encode(sentences, convert_to_tensor=True)
summary_embedding = embeddings[reference_labels == 1].mean(dim=0)

# Tính cosine similarity để chọn top-k câu
similarities = util.cos_sim(summary_embedding, embeddings)[0]
top_indices = similarities.argsort(descending=True)[:5]
predicted_labels = [1 if i in top_indices else 0 for i in range(len(sentences))]

print("Predicted summary sentences:")
for i in top_indices:
    print("-", sentences[i])


Predicted summary sentences:
- Pure justice Week notified parties that written testimony on the applicability in term of that penalty may be filed in advance of the August 9 hearing, no later than August 4.
- Our Natural Gas Distribution business earned $0.01 per share in the second quarter of both 2021 and 2020.
- Our Water Distribution business Aquarion earned $0.03 per share in the second quarters of both 2021 and 2020.
- Beginning next year, we expect Aquarion revenues to be bolstered by previously announced acquisition of New England Service Company or any SC owns the number of small water utilities that serve approximately 10,000 customers in Connecticut, Massachusetts and New Hampshire.
- We continue to expect ongoing earnings toward the lower end of our $3.81 to $3.93 per share guidance.


# Nhận xét
## 1.Với EDTSUM (abstractive summarization):
Mặc dù sử dụng mô hình mạnh như facebook/bart-large-cnn, kết quả đánh giá bằng ROUGE cho thấy hiệu suất còn khá thấp với:

ROUGE-1: 0.0305

ROUGE-2: 0.0000

ROUGE-L: 0.0208

Điều này cho thấy các mô hình tổng quát như BART chưa thực sự phù hợp để tóm tắt văn bản trong lĩnh vực tài chính, vốn chứa nhiều thuật ngữ chuyên ngành và thông tin định lượng khó xử lý. Ngoài ra, tiêu đề trong EDTSUM mang tính chất súc tích và giàu thông tin, đòi hỏi mô hình phải có hiểu biết sâu về nội dung tài chính mới có thể sinh ra tóm tắt chất lượng.

## 2.Với ECTSUM (extractive summarization):
Việc sử dụng mô hình sentence-BERT để tính độ tương đồng giữa các câu và chọn ra các câu quan trọng nhất đã giúp trích xuất được những nội dung then chốt như:

Kết quả kinh doanh của các mảng (Gas, Water)

Thông báo phiên điều trần và thời hạn

Hướng dẫn lợi nhuận và kỳ vọng tương lai

Các câu được chọn đều phản ánh đúng những nội dung cốt lõi mà một bản tóm tắt tài chính cần có, thể hiện tiềm năng áp dụng kỹ thuật extractive trong ngữ cảnh này, đặc biệt khi kết hợp với biểu diễn ngữ nghĩa mạnh như từ sentence transformers.
# Kết luận
Abstractive summarization vẫn còn hạn chế trong bối cảnh tài chính nếu không được huấn luyện chuyên biệt.

Extractive summarization, khi kết hợp với biểu diễn ngữ nghĩa, có thể cho kết quả hợp lý hơn trong việc giữ lại thông tin quan trọng từ văn bản dài.

